# BirdCLEF 2026 - CNN Training Pipeline (Multi-Model EfficientNet B0-B8, 10-Fold CV)

Training pipeline for multi-model ensemble with all EfficientNet variants.
Saves models with metadata for independent test pipeline usage.
Based on PLAN.md recommendations. Modified for OFFLINE Kaggle use.


## 1. Environment & Dependencies

In [ ]:
import os, gc, random, warnings, time, json
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import timm
from datetime import datetime

warnings.filterwarnings('ignore')
print('Libraries loaded')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')


## 2. Reproducibility & Global Config

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────────
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Config ─────────────────────────────────────────────────────────────────
CONFIG = {
    # Model backbones: all EfficientNet variants B0-B8
    'model_backbones': [
        'tf_efficientnet_b0',
        'tf_efficientnet_b1',
        'tf_efficientnet_b2',
        'tf_efficientnet_b3',
        'tf_efficientnet_b4',
        'tf_efficientnet_b5',
        'tf_efficientnet_b6',
        'tf_efficientnet_b7',
        'tf_efficientnet_b8',
    ],
    
    # Audio preprocessing
    'sr': 32_000,
    'segment_sec': 5,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 512,
    'f_min': 50,
    'f_max': 14_000,
    
    # Data filtering
    'max_clips_per_species': 30,
    'min_rating': 3.0,
    
    # Cross-validation
    'n_folds': 10,
    'seed': SEED,
    
    # Training hyperparameters
    'batch_size': 32,
    'epochs': 5,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'dropout': 0.2,
}

print(f"Config: {json.dumps(CONFIG, indent=2)}")


## 3. Paths, Directories & Kaggle Replication

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO = BASE / 'train_audio'
TRAIN_SND = BASE / 'train_soundscapes'
OUT = Path('/kaggle/working')

# Create local output directories for model saving
MODELS_DIR = OUT / 'models'
LOGS_DIR = OUT / 'logs'
METADATA_DIR = OUT / 'metadata'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Discover pretrained EfficientNet models from Kaggle dataset
PRETRAINED_MODELS_DIR = Path('/kaggle/input/models/timm/tf-efficientnet/pytorch')

# Function to discover pretrained models using glob
def discover_pretrained_models():
    """Discover all available pretrained EfficientNet models and return a mapping"""
    pretrained_map = {}
    
    if not PRETRAINED_MODELS_DIR.exists():
        print(f"⚠️  Pretrained models directory not found: {PRETRAINED_MODELS_DIR}")
        return pretrained_map
    
    # Pattern: tf-efficientnet-b{0..8}
    for model_subdir in PRETRAINED_MODELS_DIR.glob('tf-efficientnet-b*'):
        if not model_subdir.is_dir():
            continue
        
        model_name = model_subdir.name  # e.g., 'tf-efficientnet-b0'
        # Convert to timm format: tf_efficientnet_b0
        timm_name = model_name.replace('-', '_')
        
        # Find the .pth file using glob (hash may vary)
        pth_files = list(model_subdir.glob('1/*.pth'))
        
        if pth_files:
            # Use the first match (should be only one)
            pretrained_map[timm_name] = str(pth_files[0])
            print(f"✓ Found {timm_name}: {pth_files[0].name}")
    
    return pretrained_map

# Discover available pretrained models
PRETRAINED_MODELS = discover_pretrained_models()
print(f"\nDiscovered {len(PRETRAINED_MODELS)} pretrained models")

print(f"BASE: {BASE}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"Directories created successfully")

# Verify data availability
print(f"TRAIN_AUDIO exists: {TRAIN_AUDIO.exists()}")
print(f"TRAIN_SND exists: {TRAIN_SND.exists()}")


## 4. Data Loading & Audio Utilities (Replicate Original Logic)

In [ ]:
train_df = pd.read_csv(BASE / 'train.csv')
taxonomy = pd.read_csv(BASE / 'taxonomy.csv')
snd_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')

SPECIES = [c for c in sample_sub.columns if c != 'row_id']
n_classes = len(SPECIES)
species_to_idx = {sp: i for i, sp in enumerate(SPECIES)}
idx_to_species = {i: sp for i, sp in enumerate(SPECIES)}

print(f'Classes: {n_classes}')
print(f'Sample species: {SPECIES[:5]}')
print(f'Train data shape: {train_df.shape}')
print(f'Soundscape labels shape: {snd_labels.shape}')


## 4b. Prepare Data (No Leakage, True Multi-Label, Grouping)

In [ ]:
# Filter train_audio
target_set = set(SPECIES)
train_df = train_df[train_df['primary_label'].isin(target_set)].copy()
train_df = train_df[(train_df['rating'] == 0) | (train_df['rating'] >= CONFIG['min_rating'])].copy()

if CONFIG['max_clips_per_species']:
    train_df = (
        train_df.groupby('primary_label', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), CONFIG['max_clips_per_species']), random_state=CONFIG['seed']))
    ).reset_index(drop=True)

# Create targets for train_audio (single label becomes one-hot in multi-label context)
train_df['filepath'] = train_df['filename'].apply(lambda x: str(TRAIN_AUDIO / x))
train_df['group'] = train_df['filename'] # Grouping by filename
train_df['start_sec'] = 0.0 # Will dynamically slice in dataset
train_df['is_soundscape'] = False

# Soundscapes
def seconds_from_hms(hms_str):
    h, m, s = map(int, str(hms_str).split(':'))
    return h*3600 + m*60 + s

snd_labels['start_sec'] = snd_labels['start'].apply(seconds_from_hms)
snd_labels['filepath'] = snd_labels['filename'].apply(lambda x: str(TRAIN_SND / x))
snd_labels['group'] = snd_labels['filename']
snd_labels['is_soundscape'] = True

# Convert semicolon-separated primary_label to multi-label
snd_labels['labels_list'] = snd_labels['primary_label'].apply(lambda x: [sp.strip() for sp in str(x).split(';') if sp.strip() in target_set])
snd_labels['stratify_label'] = snd_labels['labels_list'].apply(lambda x: x[0] if len(x) > 0 else 'nocall')

train_df['labels_list'] = train_df['primary_label'].apply(lambda x: [x])
train_df['stratify_label'] = train_df['primary_label']

cols = ['filepath', 'start_sec', 'labels_list', 'stratify_label', 'group', 'is_soundscape']
full_df = pd.concat([train_df[cols], snd_labels[cols]], ignore_index=True)

# Build multi-label target matrix
targets = np.zeros((len(full_df), n_classes), dtype=np.float32)
for i, labels in enumerate(full_df['labels_list']):
    for sp in labels:
        targets[i, species_to_idx[sp]] = 1.0
        
full_df['target'] = list(targets)

# KFold
skf = StratifiedGroupKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['seed'])
full_df['fold'] = -1
for fold, (tr_idx, val_idx) in enumerate(skf.split(full_df, full_df['stratify_label'], groups=full_df['group'])):
    full_df.loc[val_idx, 'fold'] = fold

print("Data preparation complete")
print(f"Full dataset shape: {full_df.shape}")
print("Folds distribution:")
print(full_df['fold'].value_counts().sort_index())


## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
print("Class distribution in full dataset:")
class_counts = {}
for labels_list in full_df['labels_list']:
    for sp in labels_list:
        class_counts[sp] = class_counts.get(sp, 0) + 1

class_counts_df = pd.DataFrame(list(class_counts.items()), columns=['species', 'count']).sort_values('count', ascending=False)
print(class_counts_df.head(20))
print(f"\nTotal classes: {len(class_counts_df)}")
print(f"Average samples per class: {class_counts_df['count'].mean():.1f}")
print(f"Min samples: {class_counts_df['count'].min()}, Max samples: {class_counts_df['count'].max()}")

# Data source distribution (train audio vs soundscape)
print("\nData source distribution:")
print(f"Train audio samples: {(~full_df['is_soundscape']).sum()}")
print(f"Soundscape samples: {full_df['is_soundscape'].sum()}")

# Fold distribution
print("\nFold sizes:")
print(full_df.groupby('fold').size().describe())


## 6. Feature Engineering & Spectrogram / Augmentations

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, config, is_train=True):
        self.df = df
        self.config = config
        self.is_train = is_train
        
        self.mel_spec = T.MelSpectrogram(
            sample_rate=config['sr'],
            n_fft=config['n_fft'],
            hop_length=config['hop_length'],
            n_mels=config['n_mels'],
            f_min=config['f_min'],
            f_max=config['f_max']
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        
        # Augmentations (Time & Freq Masking)
        self.time_mask = T.TimeMasking(time_mask_param=30)
        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = row['filepath']
        is_snd = row['is_soundscape']
        
        try:
            if is_snd:
                start_frame = int(row['start_sec'] * self.config['sr'])
                frames_to_read = int(self.config['segment_sec'] * self.config['sr'])
                y, sr_orig = torchaudio.load(filepath, frame_offset=start_frame, num_frames=frames_to_read)
            else:
                y, sr_orig = torchaudio.load(filepath)
                if y.shape[1] > self.config['sr'] * self.config['segment_sec']:
                    if self.is_train:
                        start = random.randint(0, y.shape[1] - self.config['sr'] * self.config['segment_sec'])
                    else:
                        start = 0
                    y = y[:, start:start + self.config['sr'] * self.config['segment_sec']]
                
            if sr_orig != self.config['sr']:
                y = torchaudio.functional.resample(y, orig_freq=sr_orig, new_freq=self.config['sr'])
                
            if y.shape[0] > 1:
                y = y.mean(dim=0, keepdim=True)
                
            target_length = self.config['sr'] * self.config['segment_sec']
            if y.shape[1] < target_length:
                y = F.pad(y, (0, target_length - y.shape[1]))
                
        except Exception as e:
            y = torch.zeros(1, self.config['sr'] * self.config['segment_sec'])
            
        mel = self.mel_spec(y)
        mel = self.amplitude_to_db(mel)
        
        if self.is_train:
            mel = self.time_mask(mel)
            mel = self.freq_mask(mel)
            
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        mel = mel.repeat(3, 1, 1)
        
        target = torch.tensor(row['target'], dtype=torch.float32)
        return mel, target

print("BirdDataset class defined")


## 7. Model Factory: tf_efficientnet_b0..b8

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, num_classes=n_classes, model_name='tf_efficientnet_b2', config=None, pretrained=False):
        super().__init__()
        self.config = config or {}
        # Initialize without downloading from huggingface
        self.backbone = timm.create_model(model_name, pretrained=pretrained, in_chans=3)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity() # Remove original classifier
        
        dropout = self.config.get('dropout', 0.2)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        output = self.head(features)
        return output

def build_model(model_name, config):
    """Build model with pretrained weights from discovered EfficientNet models"""
    model = BirdModel(num_classes=n_classes, model_name=model_name, config=config, pretrained=False).to(device)
    
    # Attempt to load pretrained backbone weights from discovered models
    if model_name in PRETRAINED_MODELS:
        pretrained_path = PRETRAINED_MODELS[model_name]
        print(f"Loading pretrained backbone from {pretrained_path}")
        try:
            checkpoint = torch.load(pretrained_path, map_location=device)
            model.backbone.load_state_dict(checkpoint, strict=False)
            print(f"✓ Successfully loaded pretrained weights for {model_name}")
        except Exception as e:
            print(f"⚠️  Failed to load weights for {model_name}: {e}")
    else:
        print(f"⚠️  No pretrained weights found for {model_name}")
    
    return model

print("Model classes defined")


## 8. Training Loop: 10-Fold Cross-Validation per Model

In [ ]:
def train_fold(fold, model_name, config):
    """Train a single fold and save model with metadata"""
    print(f"\n{'='*60} {model_name} - Fold {fold} {'='*60}")
    
    train_df_fold = full_df[full_df['fold'] != fold].reset_index(drop=True)
    val_df_fold = full_df[full_df['fold'] == fold].reset_index(drop=True)
    
    train_dataset = BirdDataset(train_df_fold, config, is_train=True)
    val_dataset = BirdDataset(val_df_fold, config, is_train=False)
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    
    model = build_model(model_name, config)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])
    
    best_auc = 0
    best_epoch = 0
    training_history = []
    start_time = time.time()
    
    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1} Train", leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            
            out = model(x)
            loss = criterion(out, y)
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        scheduler.step()
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc=f"Epoch {epoch+1} Val", leave=False):
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                val_loss += loss.item()
                val_preds.append(torch.sigmoid(out).cpu().numpy())
                val_targets.append(y.cpu().numpy())
                
        val_preds = np.vstack(val_preds)
        val_targets = np.vstack(val_targets)
        
        aucs = []
        for i in range(n_classes):
            if val_targets[:, i].sum() > 0:
                try:
                    auc = roc_auc_score(val_targets[:, i], val_preds[:, i])
                    aucs.append(auc)
                except ValueError:
                    pass
        val_auc = np.mean(aucs) if len(aucs) > 0 else 0
        
        train_loss_avg = train_loss / len(train_loader)
        val_loss_avg = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}: Train Loss: {train_loss_avg:.4f} | Val Loss: {val_loss_avg:.4f} | Val AUC: {val_auc:.4f}")
        
        history_entry = {
            'epoch': epoch + 1,
            'train_loss': train_loss_avg,
            'val_loss': val_loss_avg,
            'val_auc': val_auc
        }
        training_history.append(history_entry)
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch + 1
            # Save checkpoint
            model_dir = MODELS_DIR / model_name / f'fold_{fold}'
            model_dir.mkdir(parents=True, exist_ok=True)
            checkpoint_path = model_dir / f'best_epoch_{epoch+1}.pth'
            torch.save(model.state_dict(), checkpoint_path)
    
    elapsed_time = time.time() - start_time
    print(f"Best Fold {fold} AUC: {best_auc:.4f} (Epoch {best_epoch}), Time: {elapsed_time:.1f}s")
    
    # Load best model and prepare for saving
    model_dir = MODELS_DIR / model_name / f'fold_{fold}'
    best_path = model_dir / f'best_epoch_{best_epoch}.pth'
    model.load_state_dict(torch.load(best_path))
    model.eval()
    
    # Get final predictions
    val_preds_final = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)
            val_preds_final.append(torch.sigmoid(out).cpu().numpy())
    val_preds_final = np.vstack(val_preds_final)
    
    return {
        'model': model,
        'predictions': val_preds_final,
        'val_df': val_df_fold,
        'best_auc': best_auc,
        'best_epoch': best_epoch,
        'history': training_history,
        'elapsed_time': elapsed_time
    }

print("Training function defined")


## 9. Metadata Saving & Model Checkpointing

In [ ]:
def save_model_with_metadata(model, fold, model_name, best_auc, best_epoch, history, elapsed_time, config):
    """Save model checkpoint and metadata JSON for later loading in test pipeline"""
    model_dir = MODELS_DIR / model_name / f'fold_{fold}'
    model_dir.mkdir(parents=True, exist_ok=True)
    
    # Save final model state
    model_path = model_dir / 'model.pth'
    torch.save(model.state_dict(), model_path)
    
    # Create metadata
    metadata = {
        'model_name': model_name,
        'fold': fold,
        'best_auc': float(best_auc),
        'best_epoch': int(best_epoch),
        'training_time_seconds': float(elapsed_time),
        'config': config,
        'n_classes': int(n_classes),
        'species': SPECIES,
        'creation_date': datetime.now().isoformat(),
        'training_history': history,
        'seed': int(config['seed']),
    }
    
    metadata_path = model_dir / 'metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Saved {model_name} fold {fold}: {model_path}")
    print(f"Saved metadata: {metadata_path}")
    
    return metadata_path, model_path

def load_model_from_checkpoint(model_name, fold):
    """Load model and metadata for inference (for test pipeline)"""
    model_dir = MODELS_DIR / model_name / f'fold_{fold}'
    model_path = model_dir / 'model.pth'
    metadata_path = model_dir / 'metadata.json'
    
    # Load metadata
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    # Rebuild model
    model = build_model(model_name, CONFIG)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, metadata

print("Metadata saving functions defined")


## 10. Execute Training: All Models x 10 Folds

In [ ]:
# Dictionary to track all trained models and their metrics
all_models_results = {}

# Train all models on all folds
for model_name in CONFIG['model_backbones']:
    print(f"\n\n{'#'*70} TRAINING {model_name} {'#'*70}\n")
    
    all_models_results[model_name] = {
        'folds': {},
        'summary': {}
    }
    
    fold_aucs = []
    
    for fold in range(CONFIG['n_folds']):
        try:
            # Train fold
            fold_result = train_fold(fold, model_name, CONFIG)
            
            # Save model with metadata
            save_model_with_metadata(
                fold_result['model'],
                fold,
                model_name,
                fold_result['best_auc'],
                fold_result['best_epoch'],
                fold_result['history'],
                fold_result['elapsed_time'],
                CONFIG
            )
            
            # Store results
            all_models_results[model_name]['folds'][fold] = {
                'best_auc': fold_result['best_auc'],
                'best_epoch': fold_result['best_epoch'],
                'time': fold_result['elapsed_time']
            }
            
            fold_aucs.append(fold_result['best_auc'])
            gc.collect()
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"ERROR training {model_name} fold {fold}: {e}")
            import traceback
            traceback.print_exc()
    
    # Summary for this model
    if fold_aucs:
        all_models_results[model_name]['summary'] = {
            'mean_auc': float(np.mean(fold_aucs)),
            'std_auc': float(np.std(fold_aucs)),
            'folds_trained': len(fold_aucs),
        }
        print(f"\n{model_name} Summary: Mean AUC = {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)


## 11. Verify Saved Models & Training Summary

In [ ]:
# Verify all models saved correctly
print("VERIFICATION: Checking saved models...\n")

saved_models_summary = []

for model_name in CONFIG['model_backbones']:
    model_dir = MODELS_DIR / model_name
    if model_dir.exists():
        fold_dirs = list(model_dir.glob('fold_*'))
        num_folds = len(fold_dirs)
        
        for fold_dir in sorted(fold_dirs):
            model_file = fold_dir / 'model.pth'
            metadata_file = fold_dir / 'metadata.json'
            
            if model_file.exists() and metadata_file.exists():
                # Load metadata to get info
                with open(metadata_file, 'r') as f:
                    meta = json.load(f)
                
                saved_models_summary.append({
                    'model': model_name,
                    'fold': meta['fold'],
                    'best_auc': meta['best_auc'],
                    'best_epoch': meta['best_epoch'],
                    'time_sec': meta['training_time_seconds'],
                    'status': '✓'
                })

# Print summary table
if saved_models_summary:
    summary_df = pd.DataFrame(saved_models_summary)
    print(summary_df.to_string(index=False))
    print(f"\nTotal saved models: {len(saved_models_summary)}")
    print(f"Expected total: {len(CONFIG['model_backbones']) * CONFIG['n_folds']}")
    
    # Summary statistics by model
    print("\n\nMean AUC by Model:")
    for model_name in CONFIG['model_backbones']:
        model_results = summary_df[summary_df['model'] == model_name]
        if len(model_results) > 0:
            print(f"  {model_name}: {model_results['best_auc'].mean():.4f} ± {model_results['best_auc'].std():.4f}")
else:
    print("No saved models found!")

# Save overall training summary
summary_file = METADATA_DIR / 'training_summary.json'
with open(summary_file, 'w') as f:
    json.dump({
        'config': CONFIG,
        'models_results': all_models_results,
        'saved_models': saved_models_summary,
        'n_classes': n_classes,
        'n_samples': len(full_df),
        'training_date': datetime.now().isoformat(),
    }, f, indent=2)

print(f"\nTraining summary saved to: {summary_file}")


## 12. Utilities: Model Discovery & Resume Training

In [ ]:
def list_all_saved_models():
    """List all saved models and their metadata"""
    models_info = []
    
    for model_dir in sorted(MODELS_DIR.glob('*')):
        if model_dir.is_dir():
            for fold_dir in sorted(model_dir.glob('fold_*')):
                metadata_file = fold_dir / 'metadata.json'
                if metadata_file.exists():
                    with open(metadata_file, 'r') as f:
                        meta = json.load(f)
                    models_info.append({
                        'model': model_dir.name,
                        'fold': meta['fold'],
                        'auc': meta['best_auc'],
                        'epoch': meta['best_epoch'],
                    })
    
    return pd.DataFrame(models_info) if models_info else pd.DataFrame()

def get_best_models_per_fold():
    """Get best model per fold (highest mean AUC across all folds)"""
    models_df = list_all_saved_models()
    if models_df.empty:
        return None
    
    fold_best = models_df.loc[models_df.groupby('fold')['auc'].idxmax()]
    return fold_best

# Show all saved models
print("All Saved Models:")
all_models = list_all_saved_models()
if not all_models.empty:
    print(all_models.to_string(index=False))
    print(f"\nTotal models: {len(all_models)}")
    print("\nBest model per fold:")
    best_per_fold = get_best_models_per_fold()
    print(best_per_fold.to_string(index=False))
else:
    print("No models saved yet")

# Save list of models to a file for easy reference
models_list = list_all_saved_models()
if not models_list.empty:
    models_list.to_csv(METADATA_DIR / 'saved_models_list.csv', index=False)
    print(f"\nSaved models list: {METADATA_DIR / 'saved_models_list.csv'}")


## Notes

### Training Pipeline Summary
This notebook trains a **multi-model ensemble** of EfficientNet variants (B0-B8) using **10-fold cross-validation**.

**Features:**
- ✅ **9 EfficientNet models**: tf_efficientnet_b0 through b8
- ✅ **10-fold CV**: Full stratified group k-fold for robust evaluation
- ✅ **Model metadata**: Each model saves a JSON with metadata for the test pipeline
- ✅ **Independent execution**: Can run without test notebook
- ✅ **Fallback support**: Test pipeline can load these models and aggregate predictions

### Output Structure
```
/kaggle/working/
├── models/
│   ├── tf_efficientnet_b0/fold_0/
│   │   ├── model.pth (weights)
│   │   ├── metadata.json (config, metrics, species info)
│   │   └── best_epoch_*.pth (checkpoints)
│   ├── tf_efficientnet_b1/fold_0/
│   │   └── ...
│   └── ... (all 9 models x 10 folds)
├── logs/
├── metadata/
│   ├── training_summary.json
│   └── saved_models_list.csv
```

### For Test Pipeline
The test notebook will:
1. Discover all saved models in `/models/`
2. Load metadata from each `metadata.json`
3. Aggregate predictions from all folds and models
4. Apply fallback logic if test files are missing

### GPU Memory Note
Training 9 models x 10 folds = 90 model trainings. Each takes ~5-10 min on standard GPU.
Adjust BATCH_SIZE or EPOCHS if running out of memory.

### Next Step
Once satisfied with results, create the **test pipeline notebook** at:
`kaggle-notebooks/cnn/birdclef-2026-cnn-test.ipynb`
